# Employee Data Warehouse with SCD Type 2
### Data Engineering Project on Databricks & Delta Lake

This project implements a Slowly Changing Dimension (SCD Type 2) pipeline to track 
historical changes in employee data (department, salary, manager, location) while 
preserving full audit history.

**Architecture:** Bronze (raw data) -> Silver (cleaned data) -> SCD Table (historical tracking)


## Step 1: Bronze Layer - Raw Data
Employee and performance data generated using Python (Faker library) and stored as Delta tables.
See notebook `01_bronze_ingestion` for data generation code.

In [0]:
%sql
SELECT * FROM employees_bronze LIMIT 10;

emp_id,name,department,salary,manager_id,location,join_date
E1001,Allison Hill,HR,59256,None,Bangalore,2022-07-30
E1002,Megan Mcclain,Operations,43434,None,Delhi,2022-01-12
E1003,Allen Robinson,Sales,107397,None,Hyderabad,2023-09-15
E1004,Cristian Santos,Sales,33905,None,Bhubaneswar,2023-11-12
E1005,Kevin Pacheco,HR,60495,None,Delhi,2021-09-16
E1006,Melissa Peterson,Marketing,33478,None,Delhi,2022-04-22
E1007,Gabrielle Davis,HR,115181,None,Delhi,2022-01-26
E1008,Lindsey Roman,Finance,58893,None,Hyderabad,2024-08-08
E1009,Valerie Gray,Marketing,66463,None,Bhubaneswar,2023-11-22
E1010,Lisa Hensley,HR,85392,None,Pune,2023-12-29


## Step 2: Silver Layer - Data Cleaning
Cleaning inconsistent values (e.g. "None" string converted to actual NULL), 
trimming whitespace, and validating records.

In [0]:
CREATE OR REPLACE TABLE employees_silver AS
SELECT
    emp_id,
    TRIM(name) AS name,
    TRIM(department) AS department,
    salary,
    CASE 
        WHEN manager_id = 'None' THEN NULL 
        ELSE manager_id 
    END AS manager_id,
    TRIM(location) AS location,
    join_date
FROM employees_bronze
WHERE emp_id IS NOT NULL;

num_affected_rows,num_inserted_rows


## Step 3: Data Quality Check - Duplicate Detection (Subquery)

In [0]:
SELECT emp_id, COUNT(*) as total_count
FROM employees_silver
WHERE emp_id IN (
    SELECT emp_id 
    FROM employees_silver 
    GROUP BY emp_id 
    HAVING COUNT(*) > 1
)
GROUP BY emp_id;

emp_id,total_count


## Step 4: SCD Type 2 Table Structure

In [0]:
CREATE OR REPLACE TABLE employees_scd (
    emp_id STRING,
    name STRING,
    department STRING,
    salary INT,
    manager_id STRING,
    location STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
) USING DELTA;

## Step 5: Initial Load into SCD Table

In [0]:
INSERT INTO employees_scd
SELECT
    emp_id,
    name,
    department,
    salary,
    manager_id,
    location,
    join_date AS start_date,
    NULL AS end_date,
    TRUE AS is_current
FROM employees_silver;

num_affected_rows,num_inserted_rows
200,200


## Step 6: Applying SCD Type 2 - MERGE Logic
When an employee's department, salary, manager, or location changes, the old record 
is closed (end_date set, is_current = false) and a new record is inserted (is_current = true).
Uses `IS DISTINCT FROM` for NULL-safe comparison.

In [0]:
MERGE INTO employees_scd AS target
USING employees_updates AS source
ON target.emp_id = source.emp_id AND target.is_current = TRUE
WHEN MATCHED AND (
    target.department IS DISTINCT FROM source.department
    OR target.salary IS DISTINCT FROM source.salary
    OR target.manager_id IS DISTINCT FROM source.manager_id
    OR target.location IS DISTINCT FROM source.location
) THEN
UPDATE SET
    target.end_date = source.change_date,
    target.is_current = FALSE;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
INSERT INTO employees_scd
SELECT emp_id, name, department, salary, manager_id, location, change_date AS start_date, NULL AS end_date, TRUE AS is_current
FROM employees_updates;

num_affected_rows,num_inserted_rows
5,5


## Step 7: Historical Analytics

In [0]:
SELECT emp_id, name, department, salary, start_date, end_date
FROM employees_scd
WHERE emp_id = 'E1001'
AND '2022-08-01' BETWEEN start_date AND COALESCE(end_date, CURRENT_DATE());

emp_id,name,department,salary,start_date,end_date
E1001,Allison Hill,HR,59256,2022-07-30,null


In [0]:
SELECT 
    emp_id, 
    name, 
    salary AS current_or_past_salary,
    start_date,
    LAG(salary) OVER (PARTITION BY emp_id ORDER BY start_date) AS previous_salary,
    salary - LAG(salary) OVER (PARTITION BY emp_id ORDER BY start_date) AS salary_change
FROM employees_scd
ORDER BY emp_id, start_date;

emp_id,name,current_or_past_salary,start_date,previous_salary,salary_change
E1001,Allison Hill,59256,2022-07-30,null,null
E1002,Megan Mcclain,43434,2022-01-12,null,null
E1002,Megan Mcclain,43434,2026-09-08,43434,0
E1003,Allen Robinson,107397,2023-09-15,null,null
E1003,Allen Robinson,122397,2026-09-08,107397,15000
E1004,Cristian Santos,33905,2023-11-12,null,null
E1004,Cristian Santos,33905,2026-09-08,33905,0
E1005,Kevin Pacheco,60495,2021-09-16,null,null
E1005,Kevin Pacheco,68495,2026-09-08,60495,8000
E1006,Melissa Peterson,33478,2022-04-22,null,null


In [0]:
SELECT department, COUNT(*) as headcount
FROM employees_scd
WHERE is_current = TRUE
GROUP BY department
ORDER BY headcount DESC;

department,headcount
Marketing,39
Finance,37
Sales,35
HR,35
Operations,34
Engineering,20
